In [1]:
# Single-cell Google Colab example
%pip install -q -U google-genai pydantic

import json
from getpass import getpass
from typing import Literal

from google import genai
from google.genai import types
from pydantic import BaseModel

#client = genai.Client(
#    api_key=getpass("Enter your Gemini API key: ")
#)

In [2]:
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

In [3]:
MODEL = "gemini-flash-latest"
MAX_STEPS = 10
TARGET = 70

# --------------------------------------------------
# 1. Simulated environment
# --------------------------------------------------
machine = {
    "temperature": 92,
    "cooling_on": False,
    "last_reading": None,
}


# --------------------------------------------------
# 2. Tools the agent can use
# --------------------------------------------------
def read_temperature():
    # Each reading represents one simulated time interval.
    # Cooling reduces temperature by 12°C per interval.
    if machine["cooling_on"]:
        machine["temperature"] = max(
            25, machine["temperature"] - 12
        )

    machine["last_reading"] = machine["temperature"]

    return {
        "temperature_c": machine["last_reading"],
        "cooling_on": machine["cooling_on"],
    }


def cooling_on():
    machine["cooling_on"] = True
    return {"cooling_on": True}


def cooling_off():
    machine["cooling_on"] = False
    return {"cooling_on": False}


TOOLS = {
    "read_temperature": read_temperature,
    "cooling_on": cooling_on,
    "cooling_off": cooling_off,
}


# --------------------------------------------------
# 3. Structured decision returned by Gemini
# --------------------------------------------------
class Decision(BaseModel):
    action: Literal[
        "read_temperature",
        "cooling_on",
        "cooling_off",
        "finish",
    ]
    explanation: str


# --------------------------------------------------
# 4. Goal and operating instructions
# --------------------------------------------------
instructions = f"""
You control a SIMULATED machine through the available actions.

GOAL:
Reduce temperature to {TARGET}°C or below, then turn cooling off.

AVAILABLE ACTIONS:
- read_temperature: Get a fresh sensor reading. Each call advances
  the simulation one interval; if cooling is on, temperature drops.
- cooling_on: Turn cooling on.
- cooling_off: Turn cooling off.
- finish: Declare the goal complete.

OPERATING RULES:
- Choose one action at a time based on actual tool results.
- Start by reading the temperature.
- After changing cooling, read the temperature to check its effect.
- Do not invent sensor readings.
- Finish only after a measured temperature is at or below
  {TARGET}°C and cooling is off.
- Keep the explanation to one short sentence.
"""

history = []


def goal_reached():
    return (
        machine["last_reading"] is not None
        and machine["last_reading"] <= TARGET
        and not machine["cooling_on"]
    )


# --------------------------------------------------
# 5. Autonomous decision -> action -> feedback loop
# --------------------------------------------------
print("GOAL: Cool the machine to 70°C or below, then turn cooling off.")

try:
    for step in range(1, MAX_STEPS + 1):
        response = client.models.generate_content(
            model=MODEL,
            contents=(
                instructions
                + "\n\nAction and observation history:\n"
                + json.dumps(history)
            ),
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=Decision,
            ),
        )

        if step == 1:
            print("Actual model used:", response.model_version)

        decision = Decision.model_validate_json(response.text or "")

        print(f"\nStep {step}: {decision.action}")
        print("Explanation:", decision.explanation)

        if decision.action == "finish":
            # Python verifies success rather than trusting the LLM.
            if goal_reached():
                print("✅ Goal verified. Agent finished.")
                break

            result = {
                "error": "Goal incomplete. Continue using tools.",
                "last_reading": machine["last_reading"],
                "cooling_on": machine["cooling_on"],
            }
        else:
            # Execute only a tool from the approved dictionary.
            result = TOOLS[decision.action]()
            print("Tool result:", result)

        history.append({
            "action": decision.action,
            "result": result,
        })
    else:
        print("\nStep limit reached.")
        print("Goal achieved:", goal_reached())

except Exception as error:
    print(f"\nAgent stopped: {type(error).__name__}: {error}")

finally:
    client.close()

GOAL: Cool the machine to 70°C or below, then turn cooling off.
Actual model used: gemini-3.8-flash

Step 1: read_temperature
Explanation: Start by reading the initial temperature.
Tool result: {'temperature_c': 92, 'cooling_on': False}

Step 2: cooling_on
Explanation: Turn cooling on to start reducing the temperature towards 70°C.
Tool result: {'cooling_on': True}

Step 3: read_temperature
Explanation: Read the temperature to monitor cooling progress after turning cooling on.
Tool result: {'temperature_c': 80, 'cooling_on': True}

Step 4: read_temperature
Explanation: Temperature is 80°C, so reading again to let cooling lower it further.
Tool result: {'temperature_c': 68, 'cooling_on': True}

Step 5: cooling_off
Explanation: The temperature is now below 70°C, so cooling can be turned off.
Tool result: {'cooling_on': False}

Step 6: read_temperature
Explanation: Read the temperature to check the machine state after turning cooling off.
Tool result: {'temperature_c': 68, 'cooling_on': F